I am using this in order just to write some pieces of code, I don't really have structure so far and am not too sure how to structure it, so I'm hoping that just by writing some stuff I make more sense of it and figure out the finer details later.

In [103]:
import numpy as np
rand = np.random.default_rng(32)

This is the matrixes that are within the numerical example:

The q matrices are the transition matrices with q_11 being the matrix for player 1 doing action 1, q_12 for player one doing action 2, q_21 is player 2 doing action 1, q_22 is player 2 doing action 2.

In [23]:
q_11 = np.matrix('-17 8 0 5 4 0; 6 -30 11 6 1 6; 0 7 -10 0 1 2; 6 5 4 -22 2 5; 6 9 8 3 -32 6; 7 5 2 0 1 -15').A

q_12 = np.matrix('-20 2 9 0 8 1; 4 -30 0 4 9 13; 1 5 -25 9 9 1; 4 6 1 -39 8 20; 8 4 1 7 -29 9; 4 2 6 0 5 -17').A

q_21 = np.matrix('-22 9 1 7 4 1; 1 -20 9 3 6 1; 1 6 -9 2 0 0; 7 7 2 -19 3 0; 2 4 7 6 -23 4; 6 6 4 4 6 -26').A

q_22 = np.matrix('-15 0 2 2 1 0; 0 -49 5 30 6 8; 1 1 -24 9 9 4; 6 1 7 -24 1 9; 18 3 5 5 -35 4; 11 15 1 3 3 -33').A

print(q_11)
print(q_12)
print(q_21)
print(q_22)

[[-17   8   0   5   4   0]
 [  6 -30  11   6   1   6]
 [  0   7 -10   0   1   2]
 [  6   5   4 -22   2   5]
 [  6   9   8   3 -32   6]
 [  7   5   2   0   1 -15]]
[[-20   2   9   0   8   1]
 [  4 -30   0   4   9  13]
 [  1   5 -25   9   9   1]
 [  4   6   1 -39   8  20]
 [  8   4   1   7 -29   9]
 [  4   2   6   0   5 -17]]
[[-22   9   1   7   4   1]
 [  1 -20   9   3   6   1]
 [  1   6  -9   2   0   0]
 [  7   7   2 -19   3   0]
 [  2   4   7   6 -23   4]
 [  6   6   4   4   6 -26]]
[[-15   0   2   2   1   0]
 [  0 -49   5  30   6   8]
 [  1   1 -24   9   9   4]
 [  6   1   7 -24   1   9]
 [ 18   3   5   5 -35   4]
 [ 11  15   1   3   3 -33]]


The d matrixes are the probabilities of which action to do from which state, with d_1 being for player 1 and d_2 being for player 2.

In [26]:
d_1 = np.matrix('0.0006, 0.9994; 0.5307, 0.4693; 0.4568, 0.5432; 0.7175, 0.2825; 0.6003, 0.3997; 0.4366, 0.5634').A

d_2 = np.matrix('0.4880 0.5120; 0.7214 0.2786; 0.4724 0.5276; 0.4379 0.5621; 0.6878 0.3122; 0.6511 0.3489').A

print(d_1)
print(d_2)

# for i in range(6):
#     print(d_1[i][0] + d_1[i][1])
#     print(d_2[i][0] + d_2[i][1])
    

[[6.000e-04 9.994e-01]
 [5.307e-01 4.693e-01]
 [4.568e-01 5.432e-01]
 [7.175e-01 2.825e-01]
 [6.003e-01 3.997e-01]
 [4.366e-01 5.634e-01]]
[[0.488  0.512 ]
 [0.7214 0.2786]
 [0.4724 0.5276]
 [0.4379 0.5621]
 [0.6878 0.3122]
 [0.6511 0.3489]]


Taking the algorithm on page 11 from the paper, this is the implementation of the different parts.

First is step 2 (step 1 is just doing it for every leader and follower):

In [ ]:
def choose_action(state, d_matrix):
    action = np.random.choice([1, 2], p=d_matrix[state])
    return action


I think for the choose_action and choose_new_state can be integrated into a player class where the player will have its d matrix and q matrices already available to them.

Step 3 (choosing new state based on action) is here, it will only be for player 1, I will make the logic behind it better later

In [104]:
def probability_array(array:np.ndarray):
    sum = array.sum()
    prob_array = np.zeros(len(array))
    for idx in range(len(array)):
        prob_array[idx] = array[idx]/sum
    return prob_array

In [105]:
def choose_new_state(state, action):
    if action == 1:
        action_array = np.delete(q_11[state], state)
    if action == 2:
        action_array = np.delete(q_12[state], state)
    prob_action_array = probability_array(action_array)
    actions = np.delete([1,2,3,4,5,6],state)
    new_state = np.random.choice(actions, p=prob_action_array)
    return new_state

choose_new_state(0, 1)

np.int64(4)

now we also need to choose the time

In [167]:
def choose_time(state, action, new_state):
    if action == 1:
        denominator = q_11[state][new_state]
    if action == 2:
        denominator = q_12[state][new_state]
    random_number = rand.random()
    while random_number == 0:
        random_number = rand.random()
    numerator = -np.log10(random_number)
    time = numerator/denominator
    return time

choose_time(0,1,3)

np.float64(0.02437654679189809)

I added a - in front of the numerator, I think thats how it should be despite it being not in the paper (as far as I can tell) cos otherwise the time is negative and that doesnt feel correct to me funnily enough

We also need to check the condition of capture. As far as I understand this, it is just checking if an attacker and a defender is at the same state at the same time. This can be done by checking every time a player changes state if someone is already there. 

I might misunderstand the time thing cos it seems more complicated in the equation (equation 36). Its a thing about the intersection of the times but I am just assuming it means they were there at the same time.

I've had a revelation on the capture conidition and the confusing time stuff. Not only do they need to be at the same place at the same time, the step they are both on needs to be the same. So if both the attacker and defender are at the same state at the same time but are on different states, then the capture condition will not be met. I think this is stupid personally because if they're both there at the same time then they should meet but hey. For the moment, I will go off the assumption that they need to be on the same step and we can choose to change it later.

I'm actually only going to do the capture thing when I have done more on the workings of the game, I am now going to make the classes for the players and the game.